In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import base64
import io

st.set_page_config(
    page_title="Воронка эффективности кампании",
    layout="wide"
)

st.title("Интерактивная воронка оценки эффективности кампании")

# -----------------------
# Встроенный демо-датафрейм (из твоего Excel)
# -----------------------

def load_demo_data():
    csv_bytes = base64.b64decode(DATA_B64.encode("ascii"))
    csv_str = csv_bytes.decode("utf-8")
    df = pd.read_csv(io.StringIO(csv_str))
    return df

# -----------------------
# Подготовка данных (общая для демо и загруженного файла)
# -----------------------

def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Медиа" not in df.columns:
        st.error("В файле нет колонки 'Медиа'. Проверь структуру отчёта.")
        return pd.DataFrame()

    df = df[df["Медиа"].notna()]
    df["Медиа"] = df["Медиа"].astype(str).str.strip()
    df = df[df["Медиа"] != "Total*"]

    rename_map = {
        "Медиа": "channel",
        "Бюджет без НДС": "budget",
        "OTS": "ots",
        "CPT OTS": "cpt_ots",
        "Охватная ёмкость в OTS 18-54 с детьми 4-15": "ots_capacity",
        "Охват @1+ , % 18 с детьми 4-15": "reach_pct",
        "Охват @1+ , в людях 18-54 с детьми 4-15": "reach_people",
        "Стоимость охвата уника 1+, руб": "cprp",
        "Частота": "freq",
        "Охватная ёмкость в людях 18-54 с детьми 4-15": "reach_capacity_people",
        "Доля от общей ёмкости в контактах": "share_ots_capacity",
        "Доля от общей ёмкости в людях": "share_reach_capacity",
        "Экон. Коэф на АКБ": "econ_coeff_akb",
        "вклад по акб": "contrib_akb",
        "стоимость вклада по акб": "cpa_akb",
        "Экон. Коэф на онлайн": "econ_coeff_online",
        "вклад по онлайн": "contrib_online",
        "стоимость вклада по онлайн": "cpa_online",
        "Экон. Коэф на апп": "econ_coeff_app",
        "вклад по апп": "contrib_app",
        "стоимость вклада по апп": "cpa_app"
    }

    df = df.rename(columns=rename_map)

    num_cols = [
        "budget", "ots", "cpt_ots", "ots_capacity",
        "reach_pct", "reach_people", "cprp", "freq",
        "reach_capacity_people", "share_ots_capacity", "share_reach_capacity",
        "econ_coeff_akb", "contrib_akb", "cpa_akb",
        "econ_coeff_online", "contrib_online", "cpa_online",
        "econ_coeff_app", "contrib_app", "cpa_app"
    ]

    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

# -----------------------
# Выбор источника данных
# -----------------------

st.sidebar.header("Источник данных")

data_source = st.sidebar.radio(
    "Откуда брать данные?",
    ["Встроенный демо-файл", "Загрузить Excel"],
    index=0
)

if data_source == "Загрузить Excel":
    uploaded_file = st.file_uploader("Загрузите Excel-файл с расчётами", type=["xlsx"])
    if uploaded_file is None:
        st.info("Загрузите файл формата .xlsx или переключитесь на встроенный демо-файл.")
        st.stop()
    df_input = pd.read_excel(uploaded_file)
else:
    df_input = load_demo_data()

df = prepare_data(df_input)
if df.empty:
    st.stop()

# -----------------------
# Фильтры
# -----------------------

if "channel" not in df.columns:
    st.error("Не найдена колонка 'Медиа' / 'channel' после обработки. Проверь файл.")
    st.stop()

channels = sorted(df["channel"].unique())
selected_channels = st.sidebar.multiselect(
    "Выберите каналы",
    options=channels,
    default=channels
)

if not selected_channels:
    st.warning("Нужно выбрать хотя бы один канал.")
    st.stop()

filtered = df[df["channel"].isin(selected_channels)]

kpi_mode = st.sidebar.radio(
    "Эконометрический KPI для основной воронки",
    options=["АКБ", "Онлайн", "App"],
    index=0
)

kpi_map_contrib = {
    "АКБ": "contrib_akb",
    "Онлайн": "contrib_online",
    "App": "contrib_app"
}
kpi_map_cpa = {
    "АКБ": "cpa_akb",
    "Онлайн": "cpa_online",
    "App": "cpa_app"
}
kpi_map_coeff = {
    "АКБ": "econ_coeff_akb",
    "Онлайн": "econ_coeff_online",
    "App": "econ_coeff_app"
}

kpi_contrib_col = kpi_map_contrib[kpi_mode]
kpi_cpa_col = kpi_map_cpa[kpi_mode]
kpi_coeff_col = kpi_map_coeff[kpi_mode]

# -----------------------
# OVERVIEW: агрегированная воронка
# -----------------------

st.subheader("Общая воронка эффективности кампании")

total_ots = filtered["ots"].sum() if "ots" in filtered.columns else 0
total_reach_people = filtered["reach_people"].sum() if "reach_people" in filtered.columns else 0
total_contrib = filtered[kpi_contrib_col].sum() if kpi_contrib_col in filtered.columns else 0

funnel_df = pd.DataFrame({
    "stage": [
        "Контакты (OTS)",
        "Охват 1+ (люди)",
        f"Вклад ({kpi_mode})"
    ],
    "value": [
        total_ots,
        total_reach_people,
        total_contrib
    ]
})

funnel_fig = px.funnel(
    funnel_df,
    x="value",
    y="stage",
    title="Воронка: от контактов к охвату и экономическому эффекту",
    labels={"value": "Значение", "stage": "Уровень"}
)
st.plotly_chart(funnel_fig, use_container_width=True)

# -----------------------
# Карточки по каналам (сводка)
# -----------------------

st.subheader("Сводка по каналам")

summary_cols = [
    "channel", "budget",
    "ots", "cpt_ots", "ots_capacity",
    "reach_pct", "reach_people", "cprp", "reach_capacity_people",
    "share_ots_capacity", "share_reach_capacity",
    "econ_coeff_akb", "contrib_akb", "cpa_akb",
    "econ_coeff_online", "contrib_online", "cpa_online",
    "econ_coeff_app", "contrib_app", "cpa_app"
]

existing_cols = [c for c in summary_cols if c in filtered.columns]
if "channel" in existing_cols:
    st.dataframe(
        filtered[existing_cols].set_index("channel"),
        use_container_width=True
    )
else:
    st.dataframe(filtered[existing_cols], use_container_width=True)

st.markdown("---")

# -----------------------
# Tabs: OTS / Reach / Экономика
# -----------------------

tab_ots, tab_reach, tab_econ = st.tabs(["Уровень 1: OTS", "Уровень 2: Охваты", "Уровень 3: Экономический вклад"])

# -----------------------
# TAB 1 — OTS
# -----------------------

with tab_ots:
    st.header("Уровень 1: Контакты (OTS)")

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### Факт OTS vs ёмкость канала")

        if {"ots", "ots_capacity"}.issubset(filtered.columns):
            df_ots = filtered.dropna(subset=["ots", "ots_capacity"])
            if not df_ots.empty:
                df_ots = df_ots.sort_values("ots")
                fig_ots_bar = px.bar(
                    df_ots,
                    x="channel",
                    y=["ots", "ots_capacity"],
                    barmode="group",
                    labels={"value": "Контакты", "channel": "Канал", "variable": "Тип"},
                )
                fig_ots_bar.update_layout(
                    legend_title_text="",
                    xaxis_title="Канал",
                    yaxis_title="Контакты"
                )
                st.plotly_chart(fig_ots_bar, use_container_width=True)
            else:
                st.info("Для выбранных каналов нет данных по OTS и ёмкости.")
        else:
            st.info("В данных отсутствуют колонки OTS и/или ёмкости OTS.")

    with col2:
        st.markdown("### CPT OTS по каналам")
        if "cpt_ots" in filtered.columns:
            df_cpt = filtered.dropna(subset=["cpt_ots"])
            if not df_cpt.empty:
                df_cpt = df_cpt.sort_values("cpt_ots")
                fig_cpt_bar = px.bar(
                    df_cpt,
                    x="channel",
                    y="cpt_ots",
                    labels={"channel": "Канал", "cpt_ots": "CPT OTS"},
                )
                st.plotly_chart(fig_cpt_bar, use_container_width=True)
            else:
                st.info("Нет данных по CPT OTS.")
        else:
            st.info("В данных нет колонки CPT OTS.")

    st.markdown("### Распределения OTS и CPT")

    col3, col4 = st.columns(2)

    with col3:
        if "ots" in filtered.columns:
            df_hist_ots = filtered.dropna(subset=["ots"])
            if not df_hist_ots.empty:
                fig_hist_ots = px.histogram(
                    df_hist_ots,
                    x="ots",
                    nbins=10,
                    labels={"ots": "OTS"},
                    title="Распределение OTS по каналам"
                )
                st.plotly_chart(fig_hist_ots, use_container_width=True)

    with col4:
        if "cpt_ots" in filtered.columns:
            df_hist_cpt = filtered.dropna(subset=["cpt_ots"])
            if not df_hist_cpt.empty:
                fig_hist_cpt = px.histogram(
                    df_hist_cpt,
                    x="cpt_ots",
                    nbins=10,
                    labels={"cpt_ots": "CPT OTS"},
                    title="Распределение CPT OTS"
                )
                st.plotly_chart(fig_hist_cpt, use_container_width=True)

    st.markdown("### Связь бюджета и OTS")

    if {"budget", "ots"}.issubset(filtered.columns):
        df_scatter_ots = filtered.dropna(subset=["budget", "ots"])
        if not df_scatter_ots.empty:
            fig_scatter_ots = px.scatter(
                df_scatter_ots,
                x="budget",
                y="ots",
                color="channel",
                size="ots",
                labels={"budget": "Бюджет без НДС", "ots": "OTS", "channel": "Канал"},
                title="Бюджет vs OTS"
            )
            st.plotly_chart(fig_scatter_ots, use_container_width=True)

# -----------------------
# TAB 2 — Reach
# -----------------------

with tab_reach:
    st.header("Уровень 2: Охваты")

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### Фактический охват 1+ vs ёмкость по людям")

        if {"reach_people", "reach_capacity_people"}.issubset(filtered.columns):
            df_reach = filtered.dropna(subset=["reach_people", "reach_capacity_people"])
            if not df_reach.empty:
                df_reach = df_reach.sort_values("reach_people")
                fig_reach_bar = px.bar(
                    df_reach,
                    x="channel",
                    y=["reach_people", "reach_capacity_people"],
                    barmode="group",
                    labels={"value": "Люди", "channel": "Канал", "variable": "Тип"},
                )
                fig_reach_bar.update_layout(
                    legend_title_text="",
                    xaxis_title="Канал",
                    yaxis_title="Человек"
                )
                st.plotly_chart(fig_reach_bar, use_container_width=True)
            else:
                st.info("Нет данных по охвату и ёмкости среди людей.")
        else:
            st.info("В данных нет колонки охвата и/или ёмкости по людям.")

    with col2:
        st.markdown("### CPRP по каналам")

        if "cprp" in filtered.columns:
            df_cprp = filtered.dropna(subset=["cprp"])
            if not df_cprp.empty:
                df_cprp = df_cprp.sort_values("cprp")
                fig_cprp_bar = px.bar(
                    df_cprp,
                    x="channel",
                    y="cprp",
                    labels={"channel": "Канал", "cprp": "Стоимость охвата уника 1+, руб"},
                )
                st.plotly_chart(fig_cprp_bar, use_container_width=True)
            else:
                st.info("Нет данных по CPRP.")
        else:
            st.info("В данных нет колонки CPRP.")

    st.markdown("### Связь бюджета и охвата 1+")

    if {"budget", "reach_people"}.issubset(filtered.columns):
        df_scatter_reach = filtered.dropna(subset=["budget", "reach_people"])
        if not df_scatter_reach.empty:
            fig_scatter_reach = px.scatter(
                df_scatter_reach,
                x="budget",
                y="reach_people",
                color="channel",
                size="reach_people",
                labels={"budget": "Бюджет без НДС", "reach_people": "Охват 1+ (люди)"},
                title="Бюджет vs Охват 1+ (люди)"
            )
            st.plotly_chart(fig_scatter_reach, use_container_width=True)

    st.markdown("### CPRP vs доля ёмкости по охвату")

    if {"cprp", "share_reach_capacity"}.issubset(filtered.columns):
        df_cprp_share = filtered.dropna(subset=["cprp", "share_reach_capacity"])
        if not df_cprp_share.empty:
            fig_cprp_share = px.scatter(
                df_cprp_share,
                x="cprp",
                y="share_reach_capacity",
                color="channel",
                labels={
                    "cprp": "CPRP (руб за 1 п.п. охвата)",
                    "share_reach_capacity": "Доля от общей ёмкости по людям"
                },
                title="CPRP vs доля ёмкости по охвату"
            )
            st.plotly_chart(fig_cprp_share, use_container_width=True)

# -----------------------
# TAB 3 — Экономический вклад
# -----------------------

with tab_econ:
    st.header("Уровень 3: Эконометрический вклад")

    st.markdown(f"Текущий KPI: **{kpi_mode}**")

    # Вклад по выбранному KPI
    col1, col2 = st.columns(2)

    with col1:
        st.markdown("### Абсолютный вклад по каналам")

        if kpi_contrib_col in filtered.columns:
            df_contrib = filtered.dropna(subset=[kpi_contrib_col])
            if not df_contrib.empty:
                df_contrib = df_contrib.sort_values(kpi_contrib_col)
                fig_contrib_bar = px.bar(
                    df_contrib,
                    x="channel",
                    y=kpi_contrib_col,
                    labels={"channel": "Канал", kpi_contrib_col: "Вклад"},
                    title=f"Вклад по {kpi_mode}"
                )
                st.plotly_chart(fig_contrib_bar, use_container_width=True)
            else:
                st.info("Нет данных по вкладу для выбранного KPI.")
        else:
            st.info("В данных нет колонки с вкладом для выбранного KPI.")

    with col2:
        st.markdown("### Стоимость вклада (CPA) по каналам")

        if kpi_cpa_col in filtered.columns:
            df_cpa = filtered.dropna(subset=[kpi_cpa_col])
            if not df_cpa.empty:
                df_cpa = df_cpa.sort_values(kpi_cpa_col)
                fig_cpa_bar = px.bar(
                    df_cpa,
                    x="channel",
                    y=kpi_cpa_col,
                    labels={"channel": "Канал", kpi_cpa_col: "Стоимость вклада"},
                    title=f"Стоимость вклада по {kpi_mode}"
                )
                st.plotly_chart(fig_cpa_bar, use_container_width=True)
            else:
                st.info("Нет данных по стоимости вклада для выбранного KPI.")
        else:
            st.info("В данных нет колонки с CPA для выбранного KPI.")

    st.markdown("### Пузырьковая диаграмма: коэффициент, вклад, бюджет")

    if {kpi_coeff_col, kpi_contrib_col, "budget"}.issubset(filtered.columns):
        df_bubble = filtered.dropna(subset=[kpi_coeff_col, kpi_contrib_col, "budget"])
        if not df_bubble.empty:
            fig_bubble = px.scatter(
                df_bubble,
                x=kpi_coeff_col,
                y=kpi_contrib_col,
                size="budget",
                color="channel",
                hover_data=["budget"],
                labels={
                    kpi_coeff_col: "Эконометрический коэффициент",
                    kpi_contrib_col: "Вклад",
                    "budget": "Бюджет"
                },
                title=f"Коэф. vs Вклад vs Бюджет ({kpi_mode})"
            )
            st.plotly_chart(fig_bubble, use_container_width=True)

    # Показать и другие KPI для сравнения
    st.markdown("### Сравнение всех трёх KPI по вкладу")

    compare_cols = {
        "Вклад АКБ": "contrib_akb",
        "Вклад онлайн": "contrib_online",
        "Вклад app": "contrib_app"
    }

    existing_compare = {label: col for label, col in compare_cols.items() if col in filtered.columns}
    if existing_compare:
        df_long = filtered[["channel"] + list(existing_compare.values())].melt(
            id_vars="channel",
            var_name="kpi",
            value_name="value"
        )
        df_long["kpi"] = df_long["kpi"].replace(existing_compare)
        df_long = df_long.dropna(subset=["value"])
        if not df_long.empty:
            fig_all_kpi = px.bar(
                df_long,
                x="channel",
                y="value",
                color="kpi",
                barmode="group",
                labels={"channel": "Канал", "value": "Вклад", "kpi": "KPI"},
                title="Сравнение вкладов по АКБ / онлайн / app"
            )
            st.plotly_chart(fig_all_kpi, use_container_width=True)

st.markdown("---")
st.caption("Прототип воронки эффективности кампании. Все расчёты строятся на загруженных данных без заглушек. Для демо доступен встроенный файл.")
